In [46]:
# https://www.youtube.com/watch?v=YUbFQlMXShY&t=2191s

In [ ]:
import numpy as np 
import pickle

In [47]:
class State:
    def __init__(self, p1, p2):
        self.board = np.zeros((BOARD_ROWS, BOARD_COLS))
        self.p1 = p1
        self.p2 = p2
        self.isEnd = False
        self.boardHash = None
        self.playerSymbol = 1

    def getHash(self):
        self.boardHash = str(self.board.reshape(BOARD_ROWS, BOARD_COLS))
        return self.boardHash

    def winner(self):
        # Rows
        for i in range(BOARD_ROWS):
            row_sum = sum(self.board[i, :])

            if row_sum == 3:
                self.isEnd = True
                return 1

            if row_sum == -3:
                self.isEnd = True
                return -1

        # Columns
        for i in range(BOARD_COLS):
            col_sum = sum(self.board[:, i])

            if col_sum == 3:
                self.isEnd = True
                return 1

            if col_sum == -3:
                self.isEnd = True
                return -1

        # Diagonals
        diag_sum1 = sum(
            self.board[i, i] for i in range(BOARD_ROWS)
        )

        diag_sum2 = sum(
            self.board[i, BOARD_COLS - i - 1]
            for i in range(BOARD_ROWS)
        )

        if diag_sum1 == 3 or diag_sum2 == 3:
            self.isEnd = True
            return 1

        if diag_sum1 == -3 or diag_sum2 == -3:
            self.isEnd = True
            return -1

        # Tie
        if len(self.availablePositions()) == 0:
            self.isEnd = True
            return 0

        # Game is not over
        return None

    def availablePositions(self):
        positions = []

        for i in range(BOARD_ROWS):
            for j in range(BOARD_COLS):
                if self.board[i, j] == 0:
                    positions.append((i, j))

        return positions

    def updateState(self, position):
        self.board[position] = self.playerSymbol

        # Switch player
        self.playerSymbol = -self.playerSymbol

    def giveReward(self):
        result = self.winner()

        if result == 1:
            self.p1.feedReward(1)
            self.p2.feedReward(0)

        elif result == -1:
            self.p1.feedReward(0)
            self.p2.feedReward(1)

        else:
            # Tie
            self.p1.feedReward(0.1)
            self.p2.feedReward(0.1)

    def reset(self):
        self.board = np.zeros((BOARD_ROWS, BOARD_COLS))
        self.boardHash = None
        self.isEnd = False
        self.playerSymbol = 1

    def play(self, rounds=100):
        for i in range(rounds):

            if i % 100 == 0:
                print("Rounds {}".format(i))

            while not self.isEnd:

                # Player 1
                positions = self.availablePositions()

                p1_action = self.p1.chooseAction(
                    positions,
                    self.board,
                    self.playerSymbol
                )

                self.updateState(p1_action)

                board_hash = self.getHash()
                self.p1.addState(board_hash)

                # Check board status
                win = self.winner()

                if win is not None:
                    self.giveReward()

                    self.p1.reset()
                    self.p2.reset()
                    self.reset()

                    break

                # Player 2
                positions = self.availablePositions()

                p2_action = self.p2.chooseAction(
                    positions,
                    self.board,
                    self.playerSymbol
                )

                self.updateState(p2_action)

                board_hash = self.getHash()
                self.p2.addState(board_hash)

                # Check board status
                win = self.winner()

                if win is not None:
                    self.giveReward()

                    self.p1.reset()
                    self.p2.reset()
                    self.reset()

                    break

    # Play against a human
    def play2(self):
        while not self.isEnd:

            # Player 1
            positions = self.availablePositions()

            p1_action = self.p1.chooseAction(
                positions,
                self.board,
                self.playerSymbol
            )

            self.updateState(p1_action)

            self.showBoard()

            win = self.winner()

            if win is not None:
                if win == 1:
                    print(self.p1.name, "wins!")
                else:
                    print("Tie!")

                self.reset()
                break

            # Player 2
            positions = self.availablePositions()

            p2_action = self.p2.chooseAction(
                positions,
                self.board,
                self.playerSymbol
            )

            self.updateState(p2_action)

            self.showBoard()

            win = self.winner()

            if win is not None:
                if win == -1:
                    print(self.p2.name, "wins!")
                else:
                    print("Tie!")

                self.reset()
                break

    def showBoard(self):
        # p1: X
        # p2: O

        for i in range(BOARD_ROWS):
            print("-------------------")

            out = "| "

            for j in range(BOARD_COLS):

                if self.board[i, j] == 1:
                    token = "X"
                elif self.board[i, j] == -1:
                    token = "O"
                else:
                    token = " "

                out += token + " | "

            print(out)

        print("-------------------")


In [48]:

class Player:
    def __init__(self, name, exp_rate=0.3):
        self.name = name
        self.states = []          # Record all positions taken
        self.lr = 0.2             # Learning rate
        self.exp_rate = exp_rate  # Exploration rate
        self.decay_gamma = 0.9    # Discount factor
        self.states_value = {}

    def getHash(self, board):
        boardHash = str(board.reshape(BOARD_COLS * BOARD_ROWS))
        return boardHash

    def chooseAction(self, positions, current_board, symbol):

        # Exploration: choose a random action
        if np.random.uniform(0, 1) <= self.exp_rate:
            idx = np.random.choice(len(positions))
            action = positions[idx]

        # Exploitation: choose the best known action
        else:
            value_max = -999
            action = None

            for p in positions:
                next_board = current_board.copy()
                next_board[p] = symbol

                next_boardHash = self.getHash(next_board)

                value = self.states_value.get(next_boardHash, 0)

                if value > value_max:
                    value_max = value
                    action = p

        return action

    def addState(self, state):
        self.states.append(state)

    def feedReward(self, reward):
        for st in reversed(self.states):

            if self.states_value.get(st) is None:
                self.states_value[st] = 0

            self.states_value[st] += self.lr * (
                self.decay_gamma * reward
                - self.states_value[st]
            )

            reward = self.states_value[st]

    def reset(self):
        self.states = []

    def savePolicy(self):
        fw = open("policy_" + str(self.name), "wb")
        pickle.dump(self.states_value, fw)
        fw.close()

    def loadPolicy(self, file):
        fr = open(file, "rb")
        self.states_value = pickle.load(fr)
        fr.close()


In [49]:
class HumanPlayer:
    def __init__(self, name):
        self.name = name

    def chooseAction(self, positions, current_board, symbol):
        while True:
            try:
                row = int(input("Input your action row (0-2): "))
                col = int(input("Input your action col (0-2): "))

                action = (row, col)

                if action in positions:
                    return action

                print("Invalid move. That position is already occupied.")

            except ValueError:
                print("Please enter numbers from 0 to 2.")

    # Append a hash state
    def addState(self, state):
        pass

    # At the end of game backpropagate and update state value
    def feedReward(self, reward):
        pass

    def reset(self):
        pass


In [50]:
BOARD_ROWS=3
BOARD_COLS=3
p1=Player("p1")
p2=Player("p2") 
st= State(p1,p2) 
print("training....")
st.play(5)

training....
Rounds 0


In [51]:
p1.savePolicy()
p2.savePolicy()

In [52]:
p1=Player('computer', exp_rate=0) 
p1.loadPolicy("policy_p1") 

p2=HumanPlayer("human")


st=State(p1,p2)
st.play2()

-------------------
| X |   |   | 
-------------------
|   |   |   | 
-------------------
|   |   |   | 
-------------------


Input your action row (0-2):  1
Input your action col (0-2):  1


-------------------
| X |   |   | 
-------------------
|   | O |   | 
-------------------
|   |   |   | 
-------------------
-------------------
| X | X |   | 
-------------------
|   | O |   | 
-------------------
|   |   |   | 
-------------------


Input your action row (0-2):  0
Input your action col (0-2):  2


-------------------
| X | X | O | 
-------------------
|   | O |   | 
-------------------
|   |   |   | 
-------------------
-------------------
| X | X | O | 
-------------------
| X | O |   | 
-------------------
|   |   |   | 
-------------------


Input your action row (0-2):  2
Input your action col (0-2):  0


-------------------
| X | X | O | 
-------------------
| X | O |   | 
-------------------
| O |   |   | 
-------------------
human wins!


In [3]:
data = {"key": "value", "number": 42}

with open("data.pkl", "wb") as file:
    pickle.dump(data, file)
with open("data.pkl", "rb") as file:
    loaded_data = pickle.load(file)
loaded_data
{'key': 'value', 'number': 42}

{'key': 'value', 'number': 42}

In [4]:
import pickle

user_settings = {
    "theme": "dark",
    "font_size": 14,
    "show_line_numbers": True
}

with open("settings.pkl", "wb") as f:
    pickle.dump(user_settings, f)


with open("settings.pkl", "rb") as f:
    loaded_settings = pickle.load(f)


print(loaded_settings)

{'theme': 'dark', 'font_size': 14, 'show_line_numbers': True}
